# Notebook 01 — Exploratory Data Analysis
**Project:** Cost-Sensitive Deep Learning for Early Clinical Deterioration Prediction in ICU Patients  
**Dataset:** WiDS Datathon 2020 (GOSSIS — 130K+ ICU visits)  
**Target:** `hospital_death` (binary — 1 = died, 0 = survived)

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded.')

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/training_v2.csv')

print(f'Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
df.head(3)

In [ ]:
# Column types overview
print('Data types:')
print(df.dtypes.value_counts())
print()
print('Sample columns per type:')
for dtype in df.dtypes.unique():
    cols = df.select_dtypes(include=[dtype]).columns.tolist()[:5]
    print(f'  {dtype}: {cols}')

## 2. Target Variable — Class Imbalance

In [ ]:
target_counts = df['hospital_death'].value_counts()
target_pct    = df['hospital_death'].value_counts(normalize=True) * 100

print('Target distribution:')
print(pd.DataFrame({'Count': target_counts, 'Percent': target_pct.round(2)}))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
axes[0].bar(['Survived (0)', 'Died (1)'], target_counts.values,
            color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Class Distribution (Count)')
axes[0].set_ylabel('Number of Patients')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 300, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(target_counts.values, labels=['Survived', 'Died'],
            autopct='%1.1f%%', colors=['steelblue', 'tomato'],
            startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Class Distribution (%)')

plt.suptitle('hospital_death — Target Variable', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/01_class_imbalance.png', bbox_inches='tight')
plt.show()

imbalance_ratio = target_counts[0] / target_counts[1]
print(f'\nImbalance ratio: {imbalance_ratio:.1f}:1 (survived:died)')
print('→ Class imbalance confirmed. Cost-sensitive learning is justified.')

## 3. Descriptive Statistics

In [ ]:
# Numerical summary
df.describe().T.round(2)

In [ ]:
# Categorical columns summary
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')
for col in cat_cols:
    print(f'\n{col}: {df[col].nunique()} unique values')
    print(df[col].value_counts().head(5))

## 4. Missing Values Analysis

In [ ]:
missing = df.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False)

print(f'Columns with missing values: {len(missing)} / {df.shape[1]}')
print(f'\nMissing > 60% (will be dropped): {(missing > 60).sum()} columns')
print(f'Missing 20–60% (impute carefully): {((missing >= 20) & (missing <= 60)).sum()} columns')
print(f'Missing < 20% (impute with median/mode): {(missing < 20).sum()} columns')

In [ ]:
# Visualize top 40 columns with most missing data
top_missing = missing.head(40)

plt.figure(figsize=(12, 10))
colors = ['tomato' if v > 60 else 'orange' if v > 20 else 'steelblue'
          for v in top_missing.values]
plt.barh(top_missing.index, top_missing.values, color=colors)
plt.axvline(60, color='tomato',  linestyle='--', linewidth=1.5, label='Drop threshold (60%)')
plt.axvline(20, color='orange',  linestyle='--', linewidth=1.5, label='Careful imputation (20%)')
plt.xlabel('Missing (%)')
plt.title('Top 40 Columns — Missing Values', fontsize=13, fontweight='bold')
plt.legend()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../outputs/01_missing_values.png', bbox_inches='tight')
plt.show()

## 5. Key Clinical Features — Distribution by Target

In [ ]:
# Key vital signs & demographics to explore
key_features = [
    'age', 'bmi',
    'heart_rate_mean', 'map_mean',            # cardiovascular
    'resprate_mean', 'spo2_mean',             # respiratory
    'temp_mean', 'glucose_mean',              # metabolic
    'd1_creatinine_max', 'd1_lactate_max'     # lab values
]

# Filter to columns that actually exist in the dataset
key_features = [c for c in key_features if c in df.columns]
print(f'Plotting {len(key_features)} features: {key_features}')

In [ ]:
n = len(key_features)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4))
axes = axes.flatten()

for i, col in enumerate(key_features):
    for label, color in zip([0, 1], ['steelblue', 'tomato']):
        subset = df[df['hospital_death'] == label][col].dropna()
        axes[i].hist(subset, bins=40, alpha=0.6, color=color,
                     label='Survived' if label == 0 else 'Died',
                     density=True)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Density')
    axes[i].legend(fontsize=8)

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Clinical Feature Distributions — Survived vs Died',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/01_feature_distributions.png', bbox_inches='tight')
plt.show()

## 6. Correlation with Target

In [ ]:
# Top features most correlated with hospital_death
num_df = df.select_dtypes(include=[np.number])
corr_with_target = num_df.corr()['hospital_death'].drop('hospital_death')
corr_with_target = corr_with_target.dropna().abs().sort_values(ascending=False)

top_corr = corr_with_target.head(25)

plt.figure(figsize=(10, 8))
colors = ['tomato' if v > 0.1 else 'steelblue' for v in top_corr.values]
plt.barh(top_corr.index, top_corr.values, color=colors)
plt.axvline(0.1, color='tomato', linestyle='--', linewidth=1.5, label='|r| = 0.10')
plt.xlabel('|Pearson Correlation|')
plt.title('Top 25 Features Correlated with hospital_death',
          fontsize=13, fontweight='bold')
plt.legend()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../outputs/01_feature_correlation.png', bbox_inches='tight')
plt.show()

print('Top 10 most correlated features:')
print(top_corr.head(10).round(4))

## 7. ICU Type & Mortality

In [ ]:
if 'icu_type' in df.columns:
    mortality_by_icu = df.groupby('icu_type')['hospital_death'].mean().sort_values(ascending=False)

    plt.figure(figsize=(10, 5))
    bars = plt.bar(mortality_by_icu.index, mortality_by_icu.values * 100,
                   color='steelblue', edgecolor='white')
    for bar, val in zip(bars, mortality_by_icu.values):
        plt.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.2, f'{val*100:.1f}%',
                 ha='center', fontsize=9)
    plt.xticks(rotation=30, ha='right')
    plt.ylabel('Mortality Rate (%)')
    plt.title('Mortality Rate by ICU Type', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/01_mortality_by_icu.png', bbox_inches='tight')
    plt.show()
else:
    print('icu_type column not found, skipping.')

## 8. APACHE Score vs Mortality

In [ ]:
apache_col = 'apache_4a_hospital_death_prob'

if apache_col in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for label, color, name in zip([0, 1], ['steelblue', 'tomato'], ['Survived', 'Died']):
        subset = df[df['hospital_death'] == label][apache_col].dropna()
        axes[0].hist(subset, bins=40, alpha=0.6, color=color, label=name, density=True)
    axes[0].set_title('APACHE-IV Death Probability Distribution')
    axes[0].set_xlabel('APACHE-IV Death Probability')
    axes[0].legend()

    # Bin APACHE score and plot mortality per bin
    df['apache_bin'] = pd.cut(df[apache_col], bins=10)
    apache_mort = df.groupby('apache_bin')['hospital_death'].mean() * 100
    axes[1].bar(range(len(apache_mort)), apache_mort.values, color='steelblue')
    axes[1].set_xticks(range(len(apache_mort)))
    axes[1].set_xticklabels([str(b) for b in apache_mort.index], rotation=45, ha='right', fontsize=7)
    axes[1].set_ylabel('Actual Mortality Rate (%)')
    axes[1].set_title('Actual Mortality Rate per APACHE Score Bin')

    plt.suptitle('APACHE-IV Score vs Mortality', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/01_apache_vs_mortality.png', bbox_inches='tight')
    plt.show()

    df.drop(columns=['apache_bin'], inplace=True)
else:
    print(f'{apache_col} not found, skipping.')

## 9. EDA Summary

In [ ]:
print('=' * 55)
print('EDA SUMMARY')
print('=' * 55)
print(f'Total patients          : {df.shape[0]:,}')
print(f'Total features          : {df.shape[1]}')
print(f'Died (target=1)         : {target_counts[1]:,} ({target_pct[1]:.1f}%)')
print(f'Survived (target=0)     : {target_counts[0]:,} ({target_pct[0]:.1f}%)')
print(f'Imbalance ratio         : ~{imbalance_ratio:.0f}:1')
print(f'Columns with missing    : {len(missing)}')
print(f'Cols to drop (>60%)     : {(missing > 60).sum()}')
print(f'Categorical cols        : {len(cat_cols)}')
print()
print('Key findings:')
print('  - Strong class imbalance → cost-sensitive approach required')
print('  - High missing rate in many features → careful imputation needed')
print('  - APACHE score, vital signs, and lab values are top predictors')
print('  - Mortality varies significantly across ICU types')
print()
print('→ Next: Notebook 02_Preprocessing.ipynb')